In [ ]:
import json
import statistics as stat
from typing import Any

with open('../scalability_bench.json') as f:
  data: list[dict[str, Any]] = json.load(f)

In [ ]:
all_lengths = [entry['length'] for entry in data]
print(f'Among {len(all_lengths)} regexes:')
short_lengths = [l for l in all_lengths if l <= 200]
print(f' <= 200: {len(short_lengths)} ({len(short_lengths) / len(all_lengths) * 100:.2f}%)')
mid_lengths = [l for l in all_lengths if 200 < l <= 1000]
print(f' (200, 1000]: {len(mid_lengths)} ({len(mid_lengths) / len(all_lengths) * 100:.2f}%)')
long_lengths = [l for l in all_lengths if l > 1000]
print(f' > 1000: {len(long_lengths)} ({len(long_lengths) / len(all_lengths) * 100:.2f}%)')
print(f' max: {max(all_lengths)}')

print(f' median: {stat.median(all_lengths)}')
print(f' 25th percentile: {stat.quantiles(all_lengths)[0]}')
print(f' 75th percentile: {stat.quantiles(all_lengths)[2]}')

In [ ]:
import matplotlib.pyplot as plt

font = {'family': 'sans-serif',
        'size': 14}
plt.rc('font', **font)

In [ ]:
fig, ax = plt.subplots()
plot = ax.violinplot(short_lengths, orientation='horizontal',
                     showmeans=True, showmedians=True, quantiles=[.25, .75])
plot['cmedians'].set_linestyle('--')
plot['cmedians'].set_color('g')
plot['cquantiles'].set_linestyle('--')
plot['cquantiles'].set_color('c')
plot['cmeans'].set_linestyle(':')
plot['cmeans'].set_color('m')
ax.grid(True)
ax.legend([plot['cmedians'], plot['cquantiles'], plot['cmeans']],
          ['50 percentile (median)', '25/75th percentile', 'Mean'],
          loc='upper right')
ax.set_xlabel('RE length')
fig.savefig('lengths_violin_plot.pdf')

print('median: {}'.format(stat.median(short_lengths)))
print('25th percentile: {}'.format(stat.quantiles(short_lengths)[0]))
print('75th percentile: {}'.format(stat.quantiles(short_lengths)[2]))
print('mean: {:.2f}'.format(stat.mean(short_lengths)))

In [ ]:
all_run_times = [t for entry in data for key in entry if key.endswith('time') for t in entry[key]]

print(f'Among {len(all_run_times)} run times:')
fast_run_times = [t for t in all_run_times if t < 1.0]
print(f' < 1 ms: {len(fast_run_times)} ({len(fast_run_times) / len(all_run_times) * 100:.2f}%)')
quick_run_times = [t for t in all_run_times if t < 10.0]
print(f' < 10 ms: {len(quick_run_times)} ({len(quick_run_times) / len(all_run_times) * 100:.2f}%)')
short_run_times = [t for t in all_run_times if t < 100.0]
print(f' < 100 ms: {len(short_run_times)} ({len(short_run_times) / len(all_run_times) * 100:.2f}%)')
long_run_times = [t for t in all_run_times if t >= 100.0]
print(f' >= 100 ms: {len(long_run_times)} ({len(long_run_times) / len(all_run_times) * 100:.2f}%)')

In [ ]:
fig, ax = plt.subplots()
plot = ax.violinplot(long_run_times, orientation='horizontal',
                     showmeans=True, showmedians=True, quantiles=[.25, .75])
plot['cmedians'].set_linestyle('--')
plot['cmedians'].set_color('g')
plot['cquantiles'].set_linestyle('--')
plot['cquantiles'].set_color('c')
plot['cmeans'].set_linestyle(':')
plot['cmeans'].set_color('m')
ax.grid(True)
ax.legend([plot['cmedians'], plot['cquantiles'], plot['cmeans']],
          ['50 percentile (median)', '25/75th percentile', 'Mean'],
          loc='upper right')
ax.set_xlabel('Time (ms)')
fig.savefig('re_ops_times_violin_plot.pdf')

print('median: {:.0f} ms'.format(stat.median(long_run_times)))
print('25th percentile: {:.0f} ms'.format(stat.quantiles(long_run_times)[0]))
print('75th percentile: {:.0f} ms'.format(stat.quantiles(long_run_times)[2]))
print('mean: {:.0f} ms'.format(stat.mean(long_run_times)))
print('max: {:.0f} ms'.format(max(long_run_times)))

In [ ]:
long_run_time_entries = []
for entry in data:
  for key in entry:
    if key.endswith('time'):
      for i, t in enumerate(entry[key]):
        if t >= 100.0:
          op = key[:-5]
          arg_key = op + ' arg'
          if arg_key in entry:
            arg = entry[arg_key][i]
          else:
            arg = None
          long_run_time_entries.append({'id': entry['id'], 'length': entry['length'],
                                        'op': op, 'arg': arg,
                                        'arg len': len(arg) if isinstance(arg, str) else None,
                                        'time': f'{t:.0f} ms'})

long_run_time_entries.sort(key=lambda x: x['id'])
for x in long_run_time_entries:
  print(x)

In [ ]:
print('range across:')
print(f'  ids: {set(x["id"] for x in long_run_time_entries)}')
print(f'  lengths: {set(x["length"] for x in long_run_time_entries)}')
print(f'  ops: {set(x["op"] for x in long_run_time_entries)}')

In [ ]:
import numpy as np


def exponential_fit(x, y):
  ln_y = np.log(y)
  coeffs = np.polyfit(x, ln_y, 1)
  b = coeffs[0]
  a = np.exp(coeffs[1])
  print(f"y = {a:.3f} * e^({b:.3f} * x)")

  plt.scatter(x, y, label='Data')
  plt.plot(x, a * np.exp(b * x), color='red', label='Fit')
  _ = plt.legend()

In [ ]:
entry = data[4303 - 1]
assert entry['id'] == 4303
ns = range(1, 11)
times = entry['take time']
exponential_fit(ns, times)

In [ ]:
entry = data[4228 - 1]
assert entry['id'] == 4228
ns = range(1, 11)
times = entry['take time']
exponential_fit(ns, times)